In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
from shapely.ops import substring

# ----------------------------------------------------
# SETTINGS
# ----------------------------------------------------

DEV_MODE = False
TEST_ROADS = [2, 3, 9]

# ----------------------------------------------------
# PATH SETUP
# ----------------------------------------------------

try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

project_root = codes_dir.parent

output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

ml_output_folder = project_root / "Data" / "ML_Output"
ml_output_folder.mkdir(exist_ok=True)

digiroad_folder = project_root / "Data" / "Digiroad"
digiroad_folder.mkdir(exist_ok=True)

# ----------------------------------------------------
# FUNCTIONS
# ----------------------------------------------------

# Cuts the road geometry to match the segment defined by the excel data.
def cut_segment(row):
    geom = row.geometry

    # Skip invalid cases
    if geom is None or geom.length == 0:
        return None

    road_start = row["road_aet"]
    road_end = row["road_let"]

    seg_start = max(row["aet"], road_start)
    seg_end = min(row["let"], road_end)

    if seg_start >= seg_end:
        return None

    # Convert to relative distance (0 → geom.length)
    rel_start = (seg_start - road_start) / (road_end - road_start)
    rel_end = (seg_end - road_start) / (road_end - road_start)

    start_dist = rel_start * geom.length
    end_dist = rel_end * geom.length

    return substring(geom, start_dist, end_dist)

# Aggregates multi-lane records by averaging the anomaly score and keeping one geometry per direction.
def aggregate_multilane(df):
    return (
        df
        .groupby(
            ["tie", "aosa", "ajorata", "road_aet", "road_let", "aet", "let"],
            as_index=False
        )
        .agg(
            geometry=("geometry", "first"),
            anomaly_score_mean=("anomaly_score", "mean"),
            anomaly_score_max=("anomaly_score", "max"),
            anomaly_score_min=("anomaly_score", "min"),
        )
    )

def aggregate_singlelane(df):
    return (
        df
        .groupby(
            ["tie", "aosa", "road_aet", "road_let", "aet", "let"],
            as_index=False
        )
        .agg(
            geometry=("geometry", "first"),
            anomaly_score_mean=("anomaly_score", "mean"),
            anomaly_score_max=("anomaly_score", "max"),
            anomaly_score_min=("anomaly_score", "min"),
        )
    )


# ----------------------------------------------------
# LOAD DATA
# ----------------------------------------------------

#excel
ef = list(ml_output_folder.glob("*.xlsx"))

print(f"Found {len(ef)} Excel files")

excel_list = []

for file in ef:
    df = pd.read_excel(
        file,
        usecols=["tie", "kaista", "ajorata", "aosa", "aet", "let", "anomaly_score"]
    )
    df["source_file"] = file.name
    excel_list.append(df)

excel = pd.concat(excel_list, ignore_index=True)
print(excel.head())
# ------------------------------------------------

# Digiroad data
gpkg_files = list(digiroad_folder.glob("*.gpkg"))

if not gpkg_files:
    raise FileNotFoundError("No .gpkg files found in Digiroad folder")

digiroad_path = gpkg_files[0]  # or choose specific one

roads = gpd.read_file(
    digiroad_path,
    layer="DR_LINKKI",
    columns=["TIENUMERO", "TIEOSANRO", "AET", "LET", "geometry", "AJORATA"]
)

print("roads row count before dropping NULL roads:", len(roads))

roads = roads.dropna(subset=["TIENUMERO"])

print("roads row count after dropping NULL roads:", len(roads))
print("----------------------------------------------")
print(roads.head())
print("----------------------------------------------")
# Normalize column names
excel = excel.rename(columns=str.lower)
roads = roads.rename(columns=str.lower)

roads = roads.rename(columns={
    "tienumero": "tie",
    "tieosanro": "aosa",
    "aet": "road_aet",
    "let": "road_let",
})

roads = roads[roads.geometry.notnull()]

print("roads", roads.columns.tolist())
print("excel", excel.columns.tolist())

# ----------------------------------------------------
# FILTER (DEV MODE)
# ----------------------------------------------------

if DEV_MODE:
    excel = excel[excel["tie"].isin(TEST_ROADS)]
    roads = roads[roads["tie"].isin(TEST_ROADS)]

# ----------------------------------------------------
# Filter roads to only include those that are in the excel dataset
# ----------------------------------------------------
print("-----------------------------------------------")
print("Filtering roads to only include those in the excel dataset...")

keys = excel[["tie", "aosa"]].drop_duplicates()

roads = roads.merge(keys, on=["tie", "aosa"], how="inner")

print(f"Roads after filtering: {len(roads)}")

# ----------------------------------------------------
# SPLIT SINGLE-LANE AND MULTI-LANE
# -----------------------------------------------------
print("-----------------------------------------------")
print("Splitting single-lane and multi-lane records...")
# ajorata = 0 means single lane, > 0 means multi-lane

excelSingleLane = excel[excel["ajorata"] == 0].copy()
excelMultiLane = excel[excel["ajorata"] > 0].copy()

print(f"Single lane records: {len(excelSingleLane)}")
print(f"Multi lane records: {len(excelMultiLane)}")

# ----------------------------------------------------
# MERGE WITH EXCEL DATA
# ----------------------------------------------------
print("-----------------------------------------------")
print("Merging roads with excel data...")

mergedSingleLane = roads.merge(excelSingleLane, on=["tie", "aosa"], how="inner")
mergedMultiLane = roads.merge(excelMultiLane, on=["tie", "aosa", "ajorata"])

mergedSingleLane = mergedSingleLane[
    (mergedSingleLane["road_aet"] <= mergedSingleLane["let"]) &
    (mergedSingleLane["road_let"] >= mergedSingleLane["aet"])
]

mergedMultiLane = mergedMultiLane[
    (mergedMultiLane["road_aet"] <= mergedMultiLane["let"]) &
    (mergedMultiLane["road_let"] >= mergedMultiLane["aet"])
]

# ----------------------------------------------------
# MULTILANE DIRECTION SEPERATION
# ----------------------------------------------------
print("-----------------------------------------------")
print("Adjusting multi-lane records to match the correct lane...")

multilane1 = mergedMultiLane[mergedMultiLane["ajorata"] == 1].copy()
multilane2 = mergedMultiLane[mergedMultiLane["ajorata"] == 2].copy()

print(f"Multi-lane records for ajorata=1: {len(multilane1)}")
print(f"Multi-lane records for ajorata=2: {len(multilane2)}")

# ----------------------------------------------------
# CUT GEOMETRIES TO MATCH EXCEL SEGMENTS
# ----------------------------------------------------
print("-----------------------------------------------")
print("Cutting geometries into 10m segments...")

mergedSingleLane["geometry"] = mergedSingleLane.apply(cut_segment, axis=1)
multilane1["geometry"] = multilane1.apply(cut_segment, axis=1)
multilane2["geometry"] = multilane2.apply(cut_segment, axis=1)

# ----------------------------------------------------
# AGGREGATE MULTI-LANE → ONE LINE PER DIRECTION
# ----------------------------------------------------
print("-----------------------------------------------")
print("Aggregating multi-lane data per direction...")

multilane1_agg = aggregate_multilane(multilane1)
multilane2_agg = aggregate_multilane(multilane2)
SingleLane_agg = aggregate_singlelane(mergedSingleLane)

# -----------------------------------------------------
# COMBINE SINGLE-LANE AND MULTI-LANE
# -----------------------------------------------------
print("-----------------------------------------------")

merged = pd.concat([SingleLane_agg, multilane1_agg, multilane2_agg], ignore_index=True) # Results in pandas DataFrame with geometry column!!!!!!!!!!!!
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=roads.crs) # Ensure the result is a GeoDataFrame with the correct CRS <-- Needed for the next null drop step and prints

# Drop nulls and duplicates that may have been created by cutting
merged = merged[merged.geometry.notnull()]

print("Total segments:", len(merged))
print("Unique road geometries", merged.geometry.nunique())
print("Unique road numbers: ", merged.tie.nunique())

# ----------------------------------------------------
# DEBUG
# ----------------------------------------------------

# test = merged[merged["tie"] == 2].sort_values("road_aet")
# print(test[["tie","aosa","road_aet","road_let","aet","let"]].head(20))

# - ------------------------------------
# DROP UNNECESSARY COLUMNS
# -------------------------------------

merged = merged.drop(
    columns=[c for c in ["road_aet", "road_let", "kaista", "anomaly_score"] if c in merged.columns]
)

# ----------------------------------------------------
# SAVE TO GEOPACKAGE
# ----------------------------------------------------
print("-----------------------------------------------")

output_path = output_folder / "road_condition_10m.gpkg"
merged.to_file(
    output_path,
    driver="GPKG",
    engine="pyogrio", # Use pyogrio for better performance. Ran into problems with fiona when saving large datasets.
    index=False
)

print(f"Saved to: {output_path}")
print("Total segments:", len(merged))
print(merged.head())
